# Resumen documento

A lo largo de este documento simulo la solución al **Minimal Maximal Matching** de varios grafos de ejemplo

In [7]:
from bloqade.analog import start, cast, load, save
import os
import matplotlib.pyplot as plt
from bokeh.io import output_notebook
from bloqade.analog.atom_arrangement import ListOfLocations, Square
from bloqade.analog import piecewise_linear, cast
import numpy as np

output_notebook()

if not os.path.isdir("data"):
    os.mkdir("data")

Loading BokehJS ...

In [8]:
import numpy as np

#K = np.log(1 + np.sqrt(2)) / 2
K=0.4406868
print(f"K_c = {K:.7f}")
print(f"8K_c = {8*K:.7f}")

# Energía
sinh_8K = np.sinh(8*K)
cosh_8K = np.cosh(8*K)
exp_8K  = np.exp(8*K)

print(f"\nsinh(8K) = {sinh_8K:.6f}")
print(f"cosh(8K) = {cosh_8K:.6f}")
print(f"exp(8K)  = {exp_8K:.6f}")

E = 8*sinh_8K / (cosh_8K + 3) /4
print(f"\n<E> = {E:.6f}")

# Magnetización
m = (exp_8K + 2) / (2*cosh_8K + 6)
print(f"<|m|> = {m:.6f}")

# Función de partición
Z = 4*cosh_8K + 12
print(f"\nZ = {Z:.6f}")

K_c = 0.4406868
8K_c = 3.5254944

sinh(8K) = 16.970564
cosh(8K) = 17.000001
exp(8K)  = 33.970565

<E> = 1.697056
<|m|> = 0.899264

Z = 80.000004


## Geometría 1: 4 vértices en línea (3 aristas en línea)

In [9]:
# 1. Geometría (Grafo de 4 vértices -> 3 aristas/átomos)
# Átomo 0 (e1): x=0, y=0
# Átomo 1 (e2): x=0, y=5.0
# Átomo 2 (e3): x=0, y=10.0
coordenadas = [(0.0, 0.0), (0.0, 5.0), (0.0, 10.0)] 
geometria = ListOfLocations(coordenadas)
geometria.show()

# 2. Parámetros Físicos
delta_end = 8.0
omega_max = 3.3
sweep_time = 2.4

C6 = 2 * np.pi * 862690
Rb = (C6 / omega_max) ** (1/6)
print(f"Radio de Bloqueo (Rb): {Rb:.2f} um")
# Verificación de distancias:
# Distancia 0-1 = 5.0 um (< Rb, hay bloqueo)
# Distancia 1-2 = 5.0 um (< Rb, hay bloqueo)
# Distancia 0-2 = 10.0 um (> Rb, no hay bloqueo)


# 3. Formas de onda del pulso láser
durations = [0.8, sweep_time, 0.8]
rabi_amplitude_values = [0.0, omega_max, omega_max, 0.0]
rabi_detuning_values = [-delta_end, -delta_end, delta_end, delta_end]

# 4. Desintonización Local (Modulación Espacial)
# Las aristas de los extremos (e1, e3) tienen una topología diferente 
# a la arista central (e2). Reflejamos esto ajustando los factores de escala.
escala_detuning = [1.0, 0.65, 1.0]

# 5. Construcción de la Secuencia
prog = (
    geometria
    .rydberg.rabi.amplitude.uniform.piecewise_linear(durations, rabi_amplitude_values)
    # Se reemplaza .uniform por .scale() para aplicar los coeficientes locales
    .detuning.scale(escala_detuning).piecewise_linear(durations, rabi_detuning_values)
)

Radio de Bloqueo (Rb): 10.86 um


## Geometría 2: Hexágono

In [10]:
# 1. Geometría Hexagonal (6 átomos)
R = 5.0
coordenadas = [
    (R * np.cos(0), R * np.sin(0)),             # Átomo 0
    (R * np.cos(np.pi/3), R * np.sin(np.pi/3)), # Átomo 1
    (R * np.cos(2*np.pi/3), R * np.sin(2*np.pi/3)), # Átomo 2
    (R * np.cos(np.pi), R * np.sin(np.pi)),     # Átomo 3
    (R * np.cos(4*np.pi/3), R * np.sin(4*np.pi/3)), # Átomo 4
    (R * np.cos(5*np.pi/3), R * np.sin(5*np.pi/3))  # Átomo 5
]
geometria = ListOfLocations(coordenadas)
#geometria.show()

# 2. Parámetros Físicos
delta_end = 8.0
omega_max = 3.3
sweep_time = 2.4

C6 = 2 * np.pi * 862690
Rb = (C6 / omega_max) ** (1/6)
print(f"Radio de Bloqueo (Rb): {Rb:.2f} um")

# 3. Formas de onda del pulso láser
durations = [0.8, sweep_time, 0.8]
rabi_amplitude_values = [0.0, omega_max, omega_max, 0.0]
rabi_detuning_values = [-delta_end, -delta_end, delta_end, delta_end]

# 4. Desintonización Local: El catalizador del cambio
# Si el array fuera [1.0, 1.0, 1.0, 1.0, 1.0, 1.0], el resultado mayoritario sería 101010.
# Al dar un peso de 1.5 a los polos opuestos, forzamos la solución minimal de tamaño 2.
escala_detuning = [1.5, 0.5, 0.5, 1.5, 0.5, 0.5]

# 5. Construcción de la Secuencia
prog = (
    geometria
    .rydberg.rabi.amplitude.uniform.piecewise_linear(durations, rabi_amplitude_values)
    .detuning.scale(escala_detuning).piecewise_linear(durations, rabi_detuning_values)
)

Radio de Bloqueo (Rb): 10.86 um


# Geometría 3: Grafo en Y
Hay 5 nodos (A,B,C,D,E). Las 4 aristas son: $e_0:(A,B)$, $e_1:(B,C)$, $e_2:(C,D)$, $e_3:(C,E)$ (El nodo C tiene tres aristas)
Cada átomo representa una arista, no un nodo!

Los posibles maximal son excitar las aristas/átomos: {$e_0,e_2$}, {$e_0,e_3} o solo {$e_1} (este es el minimal maximal)


In [11]:
# 1. Geometría (Grafo "Y" -> 4 átomos)
d = 5.0
coordenadas = [
    (-d, 0.0),                                    # Átomo 0 (e0)
    (0.0, 0.0),                                   # Átomo 1 (e1 - Centro)
    (d * np.cos(np.pi/6), d * np.sin(np.pi/6)),   # Átomo 2 (e2)
    (d * np.cos(-np.pi/6), d * np.sin(-np.pi/6))  # Átomo 3 (e3)
]
geometria = ListOfLocations(coordenadas)
geometria.show()

# 2. Parámetros Físicos
delta_end = 8.0 #Si aumenta mucho se pueden violar bloqueos de Rydberg. Para valores demasiado bajos hay átomos que no se excitan 
omega_max = 3.3 #Varía el radio de bloqueo
sweep_time = 15

C6 = 2 * np.pi * 862690
Rb = (C6 / omega_max) ** (1/6)
print(f"Radio de Bloqueo (Rb): {Rb:.2f} um")

# 3. Formas de onda
durations = [0.8, sweep_time, 0.8]
rabi_amplitude_values = [0.0, omega_max, omega_max, 0.0]
rabi_detuning_values = [-delta_end, -delta_end, delta_end, delta_end]

# 4. Detuning
# El detuning de cada qubit refleja la conectividad de los nodos de la arista
B=0.5   #Para B=0.25, los estados 1,2,3 tienen 29 resultados (de 300). Para B=0.5, solo tienen 3 resultados (de 300).
escala_detuning = [2*B, 4*B, 3*B, 3*B]   # [2B, 4B, 3B, 3B] 


# 5. Ejecución del Programa
prog = (
    geometria
    .rydberg.rabi.amplitude.uniform.piecewise_linear(durations, rabi_amplitude_values)
    .detuning.scale(escala_detuning).piecewise_linear(durations, rabi_detuning_values)
)

Radio de Bloqueo (Rb): 10.86 um


# Ejecución y visualización

In [12]:
emu_prog = prog.bloqade.python().run(shots=300)

emu_prog.report().show()

-5000000.0 4330127.018922194 -2499999.9999999995 2499999.9999999995
